# 00 · Data quality and preparation
**Goal:** check the raw data before any analysis and record every cleaning decision (rubric 3.1).
Run `src/01_build_parquet.py` and `src/02_build_features.py` first.

Principle: **nothing is deleted.** Extreme amounts and repeated payments may be deliberate anomalies, so they are flagged, not removed.

In [29]:
import sys; sys.path.insert(0, '../src')
from utils import q, rule_check, savefig, export
import numpy as np, pandas as pd, matplotlib.pyplot as plt

## 1. Size, period, missing values

In [30]:
q('''SELECT count(*) AS n_rows, min(TS_UTC) AS first_ts, max(TS_UTC) AS last_ts,
            count(DISTINCT CURRENCY_CODE) AS currencies, count(DISTINCT ORDERING_BIC8) AS ordering_bics,
            count(DISTINCT BENEFICIARY_BIC8) AS beneficiary_bics,
            count(DISTINCT ORDERING_ACCOUNT) AS ordering_accounts,
            count(DISTINCT BENEFICIARY_ACCOUNT) AS beneficiary_accounts FROM f''').T

,0
n_rows,10000000
first_ts,2023-04-30 11:22:26
last_ts,2023-07-31 09:56:01
currencies,155
ordering_bics,8392
beneficiary_bics,8266
ordering_accounts,391132
beneficiary_accounts,2590877


In [31]:
nulls = q('SELECT sum((COLUMNS(*) IS NULL)::int) FROM p').T
nulls.columns = ['n_null']; nulls[nulls.n_null > 0]

,n_null
ANOMALY_TYPE,9992994.0


Only `ANOMALY_TYPE` has NULLs: NULL means *no anomaly label was recorded*. We recode it to `NONE` in the feature table. The case study calls the labelling *optional*, so `NONE` means **unlabelled**, not verified legitimate: when we compare anomalies with normal payments we mean unlabelled ones.

## 2. Value domains and formats

In [32]:
checks = pd.concat([
    rule_check('BIC8 = 4 digits + 2 letters (country) + 2 alphanumerics', "regexp_matches(ORDERING_BIC8,'^[0-9]{4}[A-Z]{2}[A-Z0-9]{2}$') AND regexp_matches(BENEFICIARY_BIC8,'^[0-9]{4}[A-Z]{2}[A-Z0-9]{2}$')"),
    rule_check('accounts start with _FAKE_', "starts_with(ORDERING_ACCOUNT,'_FAKE_') AND starts_with(BENEFICIARY_ACCOUNT,'_FAKE_')"),
    rule_check('USD_AMOUNT > 0', 'USD_AMOUNT > 0'),
    rule_check('TOTAL_DURATION > 0', 'TOTAL_DURATION > 0'),
    rule_check('party types in {B,C}', "ORDERING_TYPE IN ('B','C') AND BENEFICIARY_TYPE IN ('B','C')"),
    rule_check('CONTINUITY_QFLAG in {Y,N}', "CONTINUITY_QFLAG IN ('Y','N')"),
    rule_check('type in {Direct,Indirect,Covered}', "TRANSACTION_TYPE_CODE IN ('Direct','Indirect','Covered')"),
    rule_check('currency code = 3 capital letters', "regexp_matches(CURRENCY_CODE,'^[A-Z]{3}$')"),
    rule_check('amount has at most 2 decimals', 'abs(USD_AMOUNT*100 - round(USD_AMOUNT*100)) < 1e-6'),
]); checks

,rule,rows_in_scope,rows_matching,match_rate_pct,exceptions
0,BIC8 = 4 digits + 2 letters (country) + 2 alph...,10000000,10000000.0,100.0000,0.0
0,accounts start with _FAKE_,10000000,10000000.0,100.0000,0.0
0,USD_AMOUNT > 0,10000000,10000000.0,100.0000,0.0
0,TOTAL_DURATION > 0,10000000,10000000.0,100.0000,0.0
0,"party types in {B,C}",10000000,10000000.0,100.0000,0.0
0,"CONTINUITY_QFLAG in {Y,N}",10000000,10000000.0,100.0000,0.0
0,"type in {Direct,Indirect,Covered}",10000000,10000000.0,100.0000,0.0
0,currency code = 3 capital letters,10000000,10000000.0,100.0000,0.0
0,amount has at most 2 decimals,10000000,9999977.0,99.9998,23.0


## 3. Duplicates and row order

In [33]:
q('''SELECT n_copies, count(*) AS groups FROM (
       SELECT count(*) AS n_copies FROM f
       GROUP BY EPOCH_SECONDS, CURRENCY_CODE, USD_AMOUNT, ORDERING_ACCOUNT, BENEFICIARY_ACCOUNT, BIC8_PATH)
     GROUP BY 1 ORDER BY 1''')

,n_copies,groups
0,1,9999998
1,2,1


In [34]:
q('SELECT corr(ROW_ID, EPOCH_SECONDS) AS corr_row_order_vs_time FROM f')

,corr_row_order_vs_time
0,-0.000272


Exactly one pair of rows is identical on these six columns; it is kept. The file is shuffled (row position is unrelated to time), so row order carries no information. Sort by `EPOCH_SECONDS` for any sequence analysis.

## 4. Outliers (flag, do not drop)

In [35]:
q('''SELECT ANOMALY, count(*) n, min(USD_AMOUNT) min_amt, median(USD_AMOUNT) median_amt, max(USD_AMOUNT) max_amt
     FROM f GROUP BY 1 ORDER BY n DESC''')

,ANOMALY,n,min_amt,median_amt,max_amt
0,NONE,9992994,0.01,3662.58,1.395062e+09
1,ROMANCE_SCAM,2476,0.01,109.45,1.233704e+05
2,FAT_FINGER_MISTAKE,1940,0.03,260241.23,1.566413e+10
3,CEO_SCAM,945,0.12,20602.13,8.848191e+07
4,DUPLICATE_PAYMENT,679,0.58,2888.63,1.000000e+07
5,ACCOUNT_REPLACEMENT_SCAM,649,2.15,3977.30,5.836230e+06
6,CUSTOMER_RECALL,317,1.87,3163.97,1.285576e+07


In [36]:
q('''SELECT ANOMALY, count(*) n FROM f WHERE USD_AMOUNT > 1e8 GROUP BY 1 ORDER BY n DESC''')

,ANOMALY,n
0,FAT_FINGER_MISTAKE,161
1,NONE,132


## 5. Internal consistency between columns

In [37]:
cons = pd.concat([
    rule_check('PARTICIPANT_COUNT = number of BICs in BIC8_PATH', 'PARTICIPANT_COUNT = N_PATH_BICS'),
    rule_check("CONTINUITY_QFLAG='N'  <=>  path contains '...'", "(CONTINUITY_QFLAG='N') = HAS_GAP"),
    rule_check('first BIC of path = ORDERING_BIC8', 'PATH_FIRST_IS_ORD'),
    rule_check('last BIC of path = BENEFICIARY_BIC8', 'PATH_LAST_IS_BEN'),
    rule_check('IBAN account country = ordering bank country', 'ORD_ACC_COUNTRY = ORD_COUNTRY', 'ORD_ACC_IS_IBAN'),
    rule_check('IBAN account country = beneficiary bank country', 'BEN_ACC_COUNTRY = BEN_COUNTRY', 'BEN_ACC_IS_IBAN'),
]); export(cons, 'dq_consistency_checks')

,rule,rows_in_scope,rows_matching,match_rate_pct,exceptions
0,PARTICIPANT_COUNT = number of BICs in BIC8_PATH,10000000,10000000.0,100.0000,0.0
0,CONTINUITY_QFLAG='N' <=> path contains '...',10000000,10000000.0,100.0000,0.0
0,first BIC of path = ORDERING_BIC8,10000000,9501601.0,95.0160,498399.0
0,last BIC of path = BENEFICIARY_BIC8,10000000,9500735.0,95.0074,499265.0
0,IBAN account country = ordering bank country,5442631,5442631.0,100.0000,0.0
0,IBAN account country = beneficiary bank country,5405035,5405035.0,100.0000,0.0


**Read this table carefully.** Rules at 100% are structural. The two endpoint rules are *not* 100%: in about 4.6% of rows each, the path does not start at `ORDERING_BIC8` (or end at `BENEFICIARY_BIC8`). These rows are kept and studied in notebook 01, because the mismatch may itself be a generated behaviour.